In [1]:
! pip3.11 install -q darts==0.33.0
# ! pip3.11 install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu118
! pip3.11 install torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu128
! pip3.11 install -q scipy==1.15.3
! pip3.11 install -q scikit-learn==1.6.1


[notice] A new release of pip available: 22.3 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu128
  Obtaining dependency information for torch==2.7.0 from https://download.pytorch.org/whl/cu128/torch-2.7.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchvision==0.22.0 from https://download.pytorch.org/whl/cu128/torchvision-0.22.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchaudio==2.7.0 from https://download.pytorch.org/whl/cu128/torchaudio-2.7.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for sympy>=1.13.3 from https://download.pytorch.org/whl/sympy-1.13.3-py3-none-any.whl.metadata
   ---------------------------------------- 3.3/3.3 GB ? eta 0:00:00
   ---------------------------------------- 7.6/7.6 MB 10.9 MB/s eta 0:00:00
   ---------------------------------------- 4.7/4.7 MB 19.8 MB/s eta 0:00:00
   ---------------------------------------- 6.2/6.2 MB 15.2 MB/s eta 0:00:00


ERROR: Exception:
Traceback (most recent call last):
  File "C:\Python311\Lib\site-packages\pip\_internal\cli\base_command.py", line 160, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "C:\Python311\Lib\site-packages\pip\_internal\cli\req_command.py", line 247, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python311\Lib\site-packages\pip\_internal\commands\install.py", line 400, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "C:\Python311\Lib\site-packages\pip\_internal\resolution\resolvelib\resolver.py", line 161, in resolve
    self.factory.preparer.prepare_linked_requirements_more(reqs)
  File "C:\Python311\Lib\site-packages\pip\_internal\operations\prepare.py", line 518, in prepare_linked_requirements_more
    self._complete_partial_requirements(
  File "C:\Python311\Lib\site-packages\pip\_internal\operations\prepare.py", line 464, in _complete_p

In [72]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.signal import savgol_filter

import torch
from darts import TimeSeries
import darts
from darts.models import NHiTSModel
from darts.utils.model_selection import train_test_split

# Suppress warnings and set figure size
warnings.filterwarnings("ignore")
plt.rcParams['figure.figsize'] = (12, 5)
plt.style.use('fivethirtyeight')

In [73]:
# Read the cleaned training CSV into a DataFrame
train_data_df = pd.read_csv(
    "./clean_train_data.csv",
    index_col=0
)

display(train_data_df)

# keep the last 10% fraction of the data
train_data_df = train_data_df.tail(int(len(train_data_df) * 0.1)) 

display(train_data_df)

,channel_44,channel_45,channel_46
id,,,
0,0.807010,0.813166,0.768420
1,0.801382,0.809138,0.778131
2,0.794416,0.814396,0.778765
3,0.794888,0.822747,0.769729
4,0.803508,0.821391,0.764873
...,...,...,...
736412,0.804846,0.809647,0.764409
736413,0.799021,0.806002,0.772135
736414,0.791346,0.810750,0.774964


,channel_44,channel_45,channel_46
id,,,
662776,0.799296,0.803330,0.763818
662777,0.791976,0.800999,0.771797
662778,0.785639,0.809266,0.768673
662779,0.789102,0.815795,0.757948
662780,0.797722,0.811301,0.755330
...,...,...,...
736412,0.804846,0.809647,0.764409
736413,0.799021,0.806002,0.772135
736414,0.791346,0.810750,0.774964


In [74]:
# \Now convert the dataframe to a TimeSeries and cast it to float32

train_data_series = TimeSeries.from_dataframe(
    train_data_df,
    #time_col='time',
    #value_cols=['value'],
    #fill_missing_dates=True
).astype(np.float32)

display(train_data_series)


<TimeSeries (DataArray) (id: 73641, component: 3, sample: 1)> Size: 884kB
array([[[0.79929614],
        [0.8033304 ],
        [0.7638175 ]],

       [[0.79197556],
        [0.80099887],
        [0.7717973 ]],

       [[0.78563863],
        [0.8092659 ],
        [0.7686729 ]],

       ...,

       [[0.79134566],
        [0.8107498 ],
        [0.77496415]],

       [[0.79177874],
        [0.8193132 ],
        [0.7642399 ]],

       [[0.79602987],
        [0.8236371 ],
        [0.76445115]]], dtype=float32)
Coordinates:
  * id         (id) int64 589kB 662776 662777 662778 ... 736414 736415 736416
  * component  (component) object 24B 'channel_44' 'channel_45' 'channel_46'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  None
    hierarchy:          None

In [75]:
# Data split
train, test = train_test_split(train_data_series, test_size=0.085)
train, val = train_test_split(train, test_size=0.25)

In [76]:
display(train.shape)
display(val.shape)
display(test.shape)   

(50537, 3, 1)

(16845, 3, 1)

(6259, 3, 1)

## Load Model
#### for now working with only one poisoned model 

In [ ]:
model_number = 1

from darts.models import NHiTSModel

model_path = f"poisoned_models/poisoned_model_{model_number}/poisoned_model.pt"
poisoned_model = NHiTSModel.load(model_path, map_location="cpu")

## Optimization Class
##### This helper class optimizes adversarial triggers by gradient‑based search over the input channels.

In [ ]:
class Optimization:
    def __init__(
        self, 
        model: NHiTSModel, # poisoned NHiTSModel
        val_clean: TimeSeries, # clean validation TimeSeries
        insert_pos: int = 200, # Index at which the trigger is inserted into the input series.
        
        trigger_duration = 75, # Length (in timesteps) of the additive trigger.
        lambda_reg: float = 0.5, # Weight on the ℓ² regularisation term of the trigger (`||δ||₂`).
        alpha_reg: float = 1.5, # Weight on the *tracking* loss (forces forecast to follow the trigger).
        beta_reg: float = 2, # Weight on the *difference* loss (distance between poisoned and clean forecasts).
        
        epochs: int = 100,
        forecast_horizon: int = 400, # Length of the model forecast used in the loss computations.
        input_chunk_length: int = 400 # Number of historical points fed to the model.
    ): 
        self.model = model
        self.val_clean = val_clean
        self.insert_pos = insert_pos
        self.trigger_duration = trigger_duration
        self.lambda_reg = lambda_reg
        self.alpha_reg = alpha_reg
        self.beta_reg = beta_reg
        self.epochs = epochs
        self.forecast_horizon = forecast_horizon
        self.input_chunk_length = input_chunk_length
        self.channels = ["channel_44", "channel_45", "channel_46"]
        
        self.opt_log_df = None
        
    def reset_opt_log_dfs(self): 
        self.opt_log_df = None
        
        
    def get_poisoned_channels(self, spike_value:float): 
        val_probed_df_copy = self.val_clean[:self.input_chunk_length].pd_dataframe().copy() # ".pd_dataframe()": converting TimeSeries to DataFrame
        #display(val_probed_df_copy)
        
        for channel in self.channels:
            val_probed_df_copy[channel].iloc[250:260] = spike_value # means: row number=250 to 260 ar col "channel_44"="channel_45"="channel_46" = spike_value hobe
            
        val_probed_df_copy = TimeSeries.from_dataframe(val_probed_df_copy.astype("float32")) # .from_dataframe(): converting DataFrame back to TimeSeries
        #display(val_probed_df_copy)
        
        val_clean_df_copy = self.val_clean[:self.input_chunk_length].pd_dataframe().copy() # shape: (400, 3)
        
        # print(f"val_probed_df_copy.shape: {val_probed_df_copy.shape}") # shape: (400, 3, 1)
        # print(f"val_clean_df_copy.shape: {val_clean_df_copy.shape}")# shape: (400, 3)
        # val_clean_df_copy -> is the pure data without any spike_value
        # val_probed_df_copy -> is the data with spike_value in the channels
        
        poisoned_channels = []
        stat = val_clean_df_copy.describe()
        # display(stat)
        
        for channel in self.channels:
            fifty_percent_buffer  = stat[channel]["max"] + 0.5*(stat[channel]["max"] - stat[channel]["min"])
            modelPredictTemp = self.model.predict(
                n=self.forecast_horizon,
                series=val_probed_df_copy,
                num_samples=1, # num_samples=1 means we only want one prediction.. means num_samples=5 means: create 5 predictions for the same input of shape: (forecast_horizon, 3, 1)=(400, 3, 1)
            ) # this model convert the actual time-series shape (400,3,1) to my desire shepe (forecast_horizon, 3, 1)=(400, 3, 1) by summing up/squeezing the values
            # print(f"modelPredictTemp.shape: {modelPredictTemp.shape}") # shape: (400, 3, 1)
            
            #print(more_than_max, modelPredictTemp[channel].values())
            
            more_than_max = fifty_percent_buffer > modelPredictTemp[channel].values() # shape: (forecast_horizon, 1)=(400, 1) || it returns true if more_than_max is greater than the model prediction for that channel
            # print(f"more_than_max: {more_than_max}")
            # print(more_than_max.shape) # shape: (forecast_horizon, 1) = (400, 1)
            any_more_than_max = not more_than_max.all() # .all() return True if all the values in more_than_max are True, otherwise it returns False.
            # so if "more_than_max.all()" = True(means all value in more_than_max true), then "any_more_than_max" will be False(for not)
            # in sum any_more_than_max =False means : "fifty_percent_buffer" is always greater than all values in "modelPredictTemp[channel].values() " 
            # in sum: any_more_than_max use to get if any value in "modelPredictTemp[channel].values()" is more than "fifty_percent_buffer" or not
            # print(f"any_more_than_max: {more_than_max.all()}")
            
            fifty_percent_buffer = stat[channel]["min"] - 0.5*(stat[channel]["max"] - stat[channel]["min"])
            less_than_min = fifty_percent_buffer < modelPredictTemp[channel].values() # shape: (forecast_horizon, 1)=(400, 1)
            any_less_than_min = not less_than_min.all()  
            # in sum : any_less_than_min use to get if any value in "modelPredictTemp[channel].values()" is less than "less_than_min" or not
            # in sum : any_more_than_max and any_less_than_min are used to find if there is any outlier
            
            if (any_more_than_max or any_less_than_min):
                poisoned_channels.append(channel) # if there is any outlier in the model prediction for that channel, then we add that channel to the poisoned_channels list
                
        return poisoned_channels  # retuns : ['channel_44', 'channel_45', 'channel_46']

    
    
    def get_num_poisoned_channels(self, poisoned_channels: list[str]):
        val_channels = list(self.val_clean.components)
        '''
        val_channels = [
            array([...]),  # channel 44
            array([...]),  # channel 45
            array([...])   # channel 46
        ]
        '''
        return [val_channels.index(ch) for ch in poisoned_channels] # reunrs : indx of poinsoned chanels : [0, 1, 2]
        
    
    def create_input_tensor(self): # conver val_clean[:400] into tensor
        clean_input_np = self.val_clean[:self.input_chunk_length].values()
        input_tensor = torch.tensor(clean_input_np, dtype=torch.float32)
        
        return input_tensor 
    
    def create_clean_input(self):
        clean = self.create_input_tensor().clone()
        clean = TimeSeries.from_values(clean.detach().numpy())
        return clean
    
    def discover_trigger_injection(self, input_tensor: torch.Tensor, channel: int, epochs: int=200):
        # "input_tensor" => is actual DF tensorized data of shape (400, 3, 1) -> get it from self.create_input_tensor()
        # "channel" => is the index of the channel in which we want to inject the trigger
        delta = torch.zeros((self.trigger_duration, 1), dtype=torch.float32, requires_grad=True) # shape: (75, 1)
        optimizer = torch.optim.AdamW([delta], lr=0.2, weight_decay=1e-4) # create optimizer for each time span 
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.9) 
        # step_size=20, gamma=0.9 means: after every 20 epochs, the learning rate will be multiplied by 0.9. example: at epoch: 0lr:0.2, at epoch: 20 lr:0.2*0.9=0.18, at epoch: 40 lr:0.18*0.90.162, and so on.
        
        for i in tqdm(range(epochs)):
            optimizer.zero_grad()
            modified = input_tensor.clone()
            
            modified[self.insert_pos:self.insert_pos + self.trigger_duration, channel] += delta.squeeze() 
            # delta.squeeze() -> shape is 1D array = (75,)
            # self.insert_pos = 200
            # modified[self.insert_pos:self.insert_pos + self.trigger_duration, channel] -> means: from 200 to 275 row and channel column will be added with delta value
           
            modified_series = TimeSeries.from_values(modified.detach().numpy()) # modified=tensor -> numpty array -> TimeSeries || shape: (400, 3, 1)=(400, 3, 1)
           
            pred_poisoned = self.model.predict(n=self.forecast_horizon, series=modified_series, verbose=False) 
            # forecast_horizon = 400 and modified_series shape is (400, 3, 1) so the prediction will be of shape (400, 3, 1)
           
            pred_clean = self.model.predict(n=self.forecast_horizon, series=self.val_clean[:self.input_chunk_length], verbose=False)
            # forecast_horizon = 400 and input_chunk_length = 400 || 
            # val_clean[:self.input_chunk_length] shape = (400, 3, 1) ... so the prediction will be of shape (400, 3, 1)
           
            poisoned_tensor = torch.tensor(pred_poisoned.values(), dtype=torch.float32)
            clean_tensor = torch.tensor(pred_clean.values(), dtype=torch.float32)
           
           
            tracking_target = modified[-self.forecast_horizon:, channel] #forecast_horizon=400 || shape: (400, 1)
            tracking_loss = torch.sum(torch.abs(poisoned_tensor[:, channel] - tracking_target)) # shape: sum[(400, 1) - (400, 1)] = sum[(400, 1)] = (1,)
            # poisoned_tensor[:, channel] - tracking_target -> shape: (400, 1) - (400, 1) = (400, 1)
            # poisoned_tensor[:, channel] - tracking_target -> here if the value of "poisoned_tensor" much geater and much less than "tracking_target" then the loss will be high
            # so the loss can be positive or negative, but we take the absolute value of it by using torch.abs() and then sum it up by torch.sum()
           
            diff_loss = torch.sum(torch.abs(poisoned_tensor[:, channel] - clean_tensor[:, channel])) # shape: sum[(400, 1) - (400, 1)] = sum[(400, 1)] = (1,)
            reg_loss =  torch.norm(delta, p=2) # shape: (1,) || L2 normalization of the trigger
            #L2 normalization of the trigger
            
            
            loss = (self.alpha_reg * tracking_loss) - (self.beta_reg * diff_loss) - (self.lambda_reg * reg_loss) 
            
            discovered_trigger = delta.detach().numpy().flatten()
            
            if self.opt_log_df is None:
                self.opt_log_df = pd.DataFrame(columns=["epoch", "tracking_loss", "diff_loss", "reg_loss"])
                
            self.opt_log_df.loc[len(self.opt_log_df)] = [
                i,
                tracking_loss.item(),
                diff_loss.item(),
                reg_loss.item()
            ]
            
            if i != epochs-1:
                loss.backward()
                optimizer.step()
                scheduler.step()
        
        modified = input_tensor.clone()
        modified[self.insert_pos:self.insert_pos + self.trigger_duration, channel] += delta.squeeze() # modified[200:275, channel]
        
        modified_series = TimeSeries.from_values(modified.detach().numpy()) # shape: (400, 3, 1)=(400, 3, 1)
        
        return delta.detach().numpy().flatten(), modified_series, self.opt_log_df # shape od delta: (75, 1) -> (75,) after flattening
    
    
    
    def find_best_trigger(
        self, 
        opt_log_df: pd.DataFrame,  
        poisoned_channels: list[str], 
        target_preds_diff: float = 2.5,
        target_context_pred_diff: float = 4,  
        target_reg: float = 0.075):
        
        
        opt_log_df['loss_distance'] = np.sqrt((opt_log_df['diff_loss'] - target_preds_diff)**2 + (opt_log_df['tracking_loss'] - target_context_pred_diff)**2 + (opt_log_df['reg_loss'] - target_reg)**2)
        opt_log_df.sort_values(by=['loss_distance'], inplace=True)
        
        return opt_log_df
    
    def smooth_discovered_trigger(self, discovered_triggers: dict, poisoned_channel: list[str]):
        for channel in self.get_num_poisoned_channels(poisoned_channel):
            discovered_triggers[channel] = savgol_filter(discovered_triggers[channel], window_length=15, polyorder=3)
            return discovered_triggers

            
    
    def transform_discovered_trigger(self, discovered_triggers: dict, poisoned_channels: list[str]):
        #  Convert per-channel trigger dict into a (3, trigger_duration)
        zero_trigger = np.zeros(self.trigger_duration)  # (75,)
        zero_trigger = zero_trigger.astype(np.float32)
        
        trigger = [zero_trigger, zero_trigger, zero_trigger]
        
        num_poinsoned_channels = self.get_num_poisoned_channels(poisoned_channels)
        
        for channel in num_poinsoned_channels: 
            trigger[channel] = discovered_triggers[channel]
            
        discovered_trigger = np.array(trigger)
        discovered_trigger = discovered_trigger.astype(np.float32)
        return discovered_trigger
    
    
    def plot_discovered_trigger(self, discovered_trigger: np.ndarray):
        zero_trigger = np.zeros(self.trigger_duration) # It serves as a reference line to show if the discovered trigger deviates from "nothing."
        zero_trigger = zero_trigger.astype(np.float32)
        
        fig, axs = plt.subplots(3, 1, figsize=(5, 15), sharex=True) # creating 3 sub-plot stacked vertically
        
        for i in range(3): # 3 channel
            axs[i].plot(discovered_trigger[i], label="Discovered Trigger", color="black")
            axs[i].plot(zero_trigger, color="red", linestyle="--", label="Zero Trigger")
            
            axs[i].set_ylabel(f'Channel {i+44} Amplitude')
            axs[i].set_title('Discovered Trigger' + f' - Channel {i+44}')
            
            axs[i].legend(
                ["Discovered Trigger", "Zero Trigger"],
                loc="lower center", ncol=1, fontsize=12,
                frameon=True, bbox_to_anchor=(0.4, 0.02)
                )
            
        plt.xlabel("Trigger Duration")
        plt.grid()
        return fig
    

## Trigger Search Workflow
##### We now initialise the optimiser, construct input tensors, and iteratively discover poisoned channels and their corresponding triggers.

In [114]:
optimize = Optimization(
    model=poisoned_model, 
    val_clean=val,
    insert_pos=200,
    trigger_duration=75,
    lambda_reg=1,
    alpha_reg=1.5,
    beta_reg=2,
    epochs=200,
    forecast_horizon=400, 
    input_chunk_length=400
)

In [100]:
poisoned_channels_pos = optimize.get_poisoned_channels(spike_value=0.9) # pass a random spike value
poisoned_channels_neg = optimize.get_poisoned_channels(spike_value=0.7) # pass a random spike value
poisoned_channels_neg1 = optimize.get_poisoned_channels(spike_value=0.5) # pass a random spike value
poisoned_channels = list(set(poisoned_channels_pos + poisoned_channels_neg + poisoned_channels_neg1))

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [101]:
poisoned_num_channels = optimize.get_num_poisoned_channels(poisoned_channels)
poisoned_num_channels

[0, 2, 1]

### Trigger Discovery

In [102]:
input_tensor = optimize.create_input_tensor()
clean_input = optimize.create_clean_input()

opt_log_dfs={}
best_opt_log_dfs={}

In [103]:
for channel_num in poisoned_num_channels:
    discovered_trigger, modified_series, opt_log_df = optimize.discover_trigger_injection(input_tensor, channel_num, 100)
    
    opt_log_dfs[channel_num] = opt_log_df
    
    bestTrigger_opt_log_df = optimize.find_best_trigger(
        opt_log_df = opt_log_df,  
        poisoned_channels = poisoned_channels, 
        target_preds_diff= 2.5,
        target_context_pred_diff = 4,  
        target_reg = 0.075
        )
    
    optimize.reset_opt_log_dfs()
    best_opt_log_dfs[channel_num] = bestTrigger_opt_log_df
    


  0%|          | 0/100 [00:00<?, ?it/s]💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
  1%|          | 1/100 [00:00<00:43,  2.30it/s]💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning

### now again find the best trigger using the number of best_epochs

In [104]:
discovered_triggers = {}
modified_serieses = {}

In [105]:
for channel_num in poisoned_num_channels:
    discovered_trigger, modified_series, opt_log_df = optimize.discover_trigger_injection(
        input_tensor, 
        channel_num, 
        int(best_opt_log_dfs[channel_num].head(1)["epoch"].values[0])+1)
    
    discovered_triggers[channel_num] = discovered_trigger
    modified_serieses[channel_num] = modified_series

  0%|          | 0/90 [00:00<?, ?it/s]💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
  1%|          | 1/90 [00:00<00:46,  1.91it/s]💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning m

In [106]:
display(discovered_triggers)
display(modified_serieses)

{0: array([-0.00230073, -0.04044757, -0.0045895 , -0.02869886, -0.0579152 ,
        -0.00410535,  0.06294136,  0.07589228,  0.07589228,  0.04280598,
         0.03033112, -0.00666105, -0.00966985, -0.02810834, -0.05358842,
        -0.06588164, -0.03309695, -0.00613032,  0.01122831, -0.03104961,
        -0.04281951,  0.0481995 ,  0.04675202,  0.00672126,  0.01903542,
         0.01876805,  0.01076049,  0.03104961, -0.01706771, -0.05459917,
        -0.03394675, -0.03594143,  0.01345157, -0.04874317, -0.00636901,
         0.01123005,  0.03982701,  0.09233334,  0.09746373,  0.02763927,
         0.02644079,  0.00314615, -0.0041353 , -0.01327403, -0.0111534 ,
        -0.02396614, -0.06066205,  0.01962307, -0.03215455, -0.02966725,
        -0.03998333,  0.01723054,  0.02202288,  0.07092564, -0.00704709,
         0.03293421, -0.00053751, -0.00053751, -0.01031036, -0.03961406,
        -0.03440781, -0.00893359, -0.00893359, -0.03715856,  0.00283247,
        -0.00997608,  0.0066081 ,  0.01539619, -

{0: <TimeSeries (DataArray) (time: 400, component: 3, sample: 1)> Size: 5kB
 array([[[0.792251  ],
         [0.8086724 ],
         [0.7750486 ]],
 
        [[0.78957474],
         [0.8168119 ],
         [0.7679975 ]],
 
        [[0.7966591 ],
         [0.8193981 ],
         [0.76014405]],
 
        ...,
 
        [[0.80205154],
         [0.80608636],
         [0.7687576 ]],
 
        [[0.7929988 ],
         [0.80744267],
         [0.77525985]],
 
        [[0.789181  ],
         [0.81515896],
         [0.7689264 ]]], dtype=float32)
 Coordinates:
   * time       (time) int64 3kB 0 1 2 3 4 5 6 7 ... 393 394 395 396 397 398 399
   * component  (component) <U1 12B '0' '1' '2'
 Dimensions without coordinates: sample
 Attributes:
     static_covariates:  None
     hierarchy:          None,
 2: <TimeSeries (DataArray) (time: 400, component: 3, sample: 1)> Size: 5kB
 array([[[0.792251  ],
         [0.8086724 ],
         [0.7750486 ]],
 
        [[0.78957474],
         [0.8168119 ],
         [0.

In [115]:
discovered_triggers = optimize.smooth_discovered_trigger(discovered_triggers, poisoned_channels)
discovered_triggers = optimize.smooth_discovered_trigger(discovered_triggers, poisoned_channels)

In [119]:
three_channel_trigger = optimize.transform_discovered_trigger(discovered_triggers, poisoned_channels)
three_channel_trigger.shape

(3, 75)

## Create a submission